# Concrete AutoEncoder Pipeline

In [ ]:
import pandas as pd
import os
import shutil


import boto3
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
import pyarrow.dataset as ds

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import numpy as np

load_dotenv()

In [ ]:
column_path = f"s3://compressive-sensing/headers.parquet"
columns =  pd.read_parquet(column_path)['0'].values

columns[:10]

In [ ]:
s3_folder = "s3://compressive-sensing/samples/"
dataset = ds.dataset(s3_folder, format="parquet")
toscore = dataset.to_table().to_pandas()

toscore.columns = columns
toscore

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(toscore[:1000], cmap="gray", aspect="auto")
plt.title("First Scan as Bandwidth vectors")
plt.ylabel("pixel")
plt.show()

In [ ]:
class ConcreteSelect(layers.Layer):
    def __init__(self, k, input_dim, temperature=0.1, **kwargs):
        super().__init__(**kwargs)
        self.k = k
        self.input_dim = input_dim
        self.temperature = temperature

    def build(self, input_shape):
        self.logits = self.add_weight(
            shape=(self.k, self.input_dim),
            initializer='glorot_uniform',
            trainable=True,
            name='logits'
        )

    def call(self, inputs, training=None):
        if training:
            uniform = tf.random.uniform(tf.shape(self.logits), minval=0, maxval=1)
            gumbel = -tf.math.log(-tf.math.log(uniform + 1e-20) + 1e-20)
            noisy_logits = (self.logits + gumbel) / self.temperature
            scores = tf.nn.softmax(noisy_logits, axis=-1)
        else:
            scores = tf.one_hot(tf.argmax(self.logits, axis=-1), depth=self.input_dim)
        return tf.matmul(inputs, tf.transpose(scores))

    def get_config(self):
        config = super().get_config()
        config.update({
            "k": self.k,
            "input_dim": self.input_dim,
            "temperature": self.temperature
        })
        return config


num_features = 297
X_scaled = toscore.dropna().values

input_dim = X_scaled.shape[1]
k = 32

input_layer = keras.Input(shape=(input_dim,))
encoded = ConcreteSelect(k=k, input_dim=input_dim)(input_layer)

decoded = layers.Dense(64, activation="relu")(encoded)
decoded = layers.Dense(128, activation="relu")(encoded)
decoded = layers.Dense(input_dim, activation="sigmoid")(decoded)

autoencoder = keras.Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(optimizer="adam", loss="mse")

cae_hist = autoencoder.fit(
    X_scaled, X_scaled,
    epochs=50,
    batch_size=32,
    shuffle=True,
    validation_split=0.2
)

encoder = keras.Model(inputs=input_layer, outputs=encoded)
X_encoded = encoder.predict(X_scaled)

# Reconstruct
X_reconstructed = autoencoder.predict(X_scaled)

## Store Model Results

In [ ]:
validation_path = f"s3://compressive-sensing-model-results/auto-encoder/v1/score.parquet"

df = pd.DataFrame(cae_hist.history["val_loss"])
df.to_parquet(
      validation_path,
      index=False,
      engine="pyarrow",
  )

os.makedirs("artifacts", exist_ok=True)
autoencoder.save_weights("artifacts/ae_weights.weights.h5")

#### Validation only - same weights

In [ ]:
zip_path = shutil.make_archive("ae_savedmodel", "zip", "artifacts")
s3 = boto3.client("s3")
s3.upload_file(zip_path, "compressive-sensing-model-results", "auto-encoder/v1/ae_savedmodel.zip")
input_layer = keras.Input(shape=(input_dim,))
encoded = ConcreteSelect(k=k, input_dim=input_dim)(input_layer)

decoded = layers.Dense(64, activation="relu")(encoded)
decoded = layers.Dense(128, activation="relu")(encoded)
decoded = layers.Dense(input_dim, activation="sigmoid")(decoded)

autoencoder3 = keras.Model(inputs=input_layer, outputs=decoded)
autoencoder3.compile(optimizer="adam", loss="mse")
autoencoder3.load_weights("ae2/ae_weights.weights.h5")

same = np.allclose(X_reconstructed, autoencoder3.predict(X_scaled), rtol=1e-5, atol=1e-8)
print("Predictions identical within tolerance:", same)